# Challenge 5 — Orchestrator-Worker with Send API (Reference Implementation)

**Purpose:** Complete reference solution for the Orchestrator-Worker pattern using LangGraph's Send API.

**Original Challenge Notebook:** `Orchestrator-Worker with Send API.ipynb` (keep that one for practice)

---

## What This Builds

A research agent that:
1. Takes a subject (e.g., "AI Engineering")
2. Orchestrator generates 3 subtopics
3. Send API dispatches each subtopic to parallel research workers
4. Each worker writes a short paragraph
5. Combiner merges all results into one final report

---

## Key Constraints (from challenge)

- ✅ Use `Send` from `langgraph.types` — not regular edges for parallel part
- ✅ Use `Annotated[list, operator.add]` as reducer for collecting worker results
- ✅ Orchestrator and worker must have different state schemas — use subgraph pattern

## Web References & Resources

### Primary References
- **Soba Labs - Orchestrator-Worker Pattern** — Complete working implementation with Send API, Annotated reducers, and separate state schemas
  🔗 `https://github.com/soba-labs/langchain-agent-skills/blob/main/patterns/orchestrator-worker.md`
- **LangGraph Fundamentals (Official)** — Fan-out pattern with Send API and operator.add reducer
  🔗 `https://github.com/langchain-ai/langgraph-fundamentals`
- **Samanosukeh/langgraph-ticket-triage** — Production example with fan-out via Send to parallel subgraphs
  🔗 `https://github.com/Samanosukeh/langgraph-ticket-triage`
- **LangChain Academy L2 Edges Notebook** — Official course material on parallel execution
  🔗 `https://github.com/langchain-ai/lca-langgraph-essentials/blob/main/L2_edges.ipynb`

### Key Concepts from References

| Concept | Description |
|---------|-------------|
| **Send API** | True graph-level parallelism — dispatches to multiple nodes simultaneously |
| **Annotated[list, operator.add]** | Reducer that accumulates results from parallel branches |
| **Subgraph with different schema** | Worker has isolated state; parent transforms state before/after invocation |
| **input_schema / output_schema** | Explicit schemas on StateGraph for subgraph boundaries |

---

## Implementation

In [1]:
# Setup
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Send, interrupt, Command
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from typing import TypedDict, Annotated, List
import operator
import uuid

load_dotenv()

True

In [2]:
# Initialize LLM
llm = init_chat_model(
    model="qwen/qwen3.8-27b",       # The specific Groq model ID
    model_provider="groq",        # Specifies the provider
    temperature=0                 # Optional parameters
)

## State Definitions

Following the subgraph pattern: **Orchestrator and Worker have DIFFERENT state schemas**.

In [3]:
# ============================================================
# WORKER STATE (Subgraph) - Different from Orchestrator state
# ============================================================
class WorkerInput(TypedDict):
    """Input that the worker subgraph receives from Send."""
    subtopic: str
    subject: str  # Original subject for context

class WorkerOutput(TypedDict):
    """Output that the worker subgraph returns."""
    research_result: str
    subtopic: str

class WorkerState(WorkerInput, WorkerOutput):
    """Full worker state = input + output."""
    pass

# ============================================================
# ORCHESTRATOR STATE (Parent Graph) - Different from Worker state
# ============================================================
class OrchestratorState(TypedDict):
    """State for the orchestrator node."""
    subject: str
    subtopics: List[str]
    # Accumulator: collects results from parallel workers via Send
    worker_results: Annotated[List[dict], operator.add]
    final_report: str


## Worker Subgraph

The worker is a **separate compiled subgraph** with its own state schema (`WorkerState`).
It receives `WorkerInput` and returns `WorkerOutput`.

In [4]:
# ============================================================
# WORKER SUBGRAPH
# ============================================================

def research_worker(state: WorkerState) -> WorkerOutput:
    """Worker node: researches a subtopic and returns a paragraph."""
    subtopic = state["subtopic"]
    subject = state["subject"]
    
    prompt = f"""You are a research assistant. Write a concise, informative paragraph (3-5 sentences) about:
    Subtopic: {subtopic}
    Context: Part of research on "{subject}"
    
    Focus on key concepts, current trends, and practical applications.
    Be specific and technical."""
    
    response = llm.invoke([
        SystemMessage(content="You are a knowledgeable research assistant."),
        HumanMessage(content=prompt)
    ])
    
    return {
        "research_result": response.content,
        "subtopic": subtopic
    }

# Build worker subgraph with explicit input/output schemas
worker_builder = StateGraph(
    WorkerState,
    input_schema=WorkerInput,
    output_schema=WorkerOutput
)
worker_builder.add_node("research", research_worker)
worker_builder.add_edge(START, "research")
worker_builder.add_edge("research", END)

# Compile the worker subgraph
worker_subgraph = worker_builder.compile()

# Test the worker subgraph independently
# test_result = worker_subgraph.invoke({"subtopic": "RAG systems", "subject": "AI Engineering"})
# print(test_result)

## Orchestrator Node

Generates 3 subtopics from the subject.

In [5]:
# ============================================================
# ORCHESTRATOR NODE
# ============================================================

def orchestrator(state: OrchestratorState) -> dict:
    """Generates 3 subtopics from the subject."""
    subject = state["subject"]
    
    prompt = f"""Given the subject: "{subject}"
    Generate exactly 3 distinct, specific subtopics that would be valuable to research.
    Return as a JSON list of strings, e.g., ["subtopic 1", "subtopic 2", "subtopic 3"]
    Make them technical and non-overlapping."""
    
    response = llm.invoke([
        SystemMessage(content="You generate focused research subtopics. Output only a JSON list."),
        HumanMessage(content=prompt)
    ])
    
    import json
    try:
        subtopics = json.loads(response.content.strip())
        if not isinstance(subtopics, list) or len(subtopics) != 3:
            raise ValueError("Expected 3 subtopics")
    except:
        # Fallback subtopics
        subtopics = [
            f"Core concepts in {subject}",
            f"Current trends in {subject}",
            f"Practical applications of {subject}"
        ]
    
    return {"subtopics": subtopics}

## Dispatch Function (Fan-out with Send)

This is the **key function** that uses `Send` API to dispatch to parallel workers.
It returns a list of `Send` objects — one per subtopic.

In [6]:
# ============================================================
# DISPATCH FUNCTION — Uses Send API for Parallel Fan-out
# ============================================================

def dispatch_workers(state: OrchestratorState) -> List[Send]:
    """
    Fan-out: Create one Send per subtopic to the worker subgraph.
    Each Send carries the WorkerInput schema (subtopic + subject).
    """
    subtopics = state["subtopics"]
    subject = state["subject"]
    
    sends = []
    for subtopic in subtopics:
        # Send to the worker subgraph node (named "worker")
        # Payload matches WorkerInput schema
        sends.append(
            Send(
                "worker",  # Node name in parent graph
                {
                    "subtopic": subtopic,
                    "subject": subject
                }
            )
        )
    
    return sends

## Combiner Node

Merges all worker results into a final report. The `worker_results` field uses `Annotated[List[dict], operator.add]` so results from parallel workers accumulate automatically.

In [7]:
# ============================================================
# COMBINER NODE — Merges Parallel Results
# ============================================================

def combiner(state: OrchestratorState) -> dict:
    """Combines all worker results into a final report."""
    worker_results = state.get("worker_results", [])
    subject = state["subject"]
    
    if not worker_results:
        return {"final_report": f"No research results for {subject}."}
    
    # Format results for the LLM
    research_sections = []
    for result in worker_results:
        research_sections.append(f"### {result['subtopic']}\n{result['research_result']}")
    
    combined_research = "\n\n".join(research_sections)
    
    prompt = f"""Create a cohesive research report on "{subject}" from these sections:
    
    {combined_research}
    
    Write a well-structured report with:
    - Executive summary (2-3 sentences)
    - Key findings from each subtopic
    - Conclusion
    Keep it concise but comprehensive."""
    
    response = llm.invoke([
        SystemMessage(content="You synthesize research into clear reports."),
        HumanMessage(content=prompt)
    ])
    
    return {"final_report": response.content}

## Parent Graph Assembly

Wire everything together:
1. `orchestrator` node → generates subtopics
2. `dispatch_workers` conditional edges → fan-out via Send to `worker`
3. `worker` node is the **compiled worker subgraph**
4. `worker` → `combiner` (auto-converges when all parallel branches complete)
5. `combiner` → END

In [8]:
# ============================================================
# PARENT GRAPH — Orchestrator + Worker Subgraph + Combiner
# ============================================================

parent_builder = StateGraph(OrchestratorState)

# Add nodes
parent_builder.add_node("orchestrator", orchestrator)
parent_builder.add_node("worker", worker_subgraph)  # Compiled subgraph as node
parent_builder.add_node("combiner", combiner)

# Entry point
parent_builder.add_edge(START, "orchestrator")

# Fan-out: orchestrator → parallel workers via Send
parent_builder.add_conditional_edges(
    "orchestrator",
    dispatch_workers,
    # The conditional edge returns Send objects, not a string
)

# Fan-in: all workers converge to combiner automatically
parent_builder.add_edge("worker", "combiner")
parent_builder.add_edge("combiner", END)

# Compile
research_agent = parent_builder.compile()

# Visualize
# display(Image(research_agent.get_graph().draw_mermaid_png()))

## Run the Agent

In [9]:
# ============================================================
# RUN THE RESEARCH AGENT
# ============================================================

def run_research(subject: str) -> str:
    """Run the full orchestrator-worker pipeline."""
    initial_state = {
        "subject": subject,
        "subtopics": [],
        "worker_results": [],  # Will be populated by operator.add reducer
        "final_report": ""
    }
    
    result = research_agent.invoke(initial_state)
    return result["final_report"]

# Example run
# report = run_research("AI Engineering")
# print(report)

In [ ]:
run_research("Subtopics: Chunking, Embeddings, Vector DBs, Reranking, Evaluation")

'No research results for jo.'

## Architecture Diagram (Mermaid)

```mermaid
graph TD
    START --> Orchestrator
    Orchestrator -.->|Send: subtopic 1| Worker1[Worker Subgraph]
    Orchestrator -.->|Send: subtopic 2| Worker2[Worker Subgraph]
    Orchestrator -.->|Send: subtopic 3| Worker3[Worker Subgraph]
    Worker1 --> Combiner
    Worker2 --> Combiner
    Worker3 --> Combiner
    Combiner --> END
    
    classDef subgraph fill:#f9f,stroke:#333;
    class Worker1,Worker2,Worker3 subgraph;
```

---

## Key Implementation Notes

### Why Subgraph Pattern?
- **Worker state (`WorkerState`) ≠ Orchestrator state (`OrchestratorState`)**
- Worker only needs `subtopic` + `subject` — doesn't need `worker_results` accumulator
- Parent graph handles accumulation via `Annotated[List[dict], operator.add]`
- `input_schema`/`output_schema` on `StateGraph` enforces boundaries

### Why Send API?
- Regular edges = sequential or fixed fan-out
- `Send` = dynamic, graph-level parallelism determined at runtime
- Each `Send` can carry different payload (different subtopic)
- All `Send` targets execute in parallel automatically

### Why Annotated[list, operator.add]?
- LangGraph's reducer mechanism for merging parallel branch results
- `operator.add` concatenates lists from each worker
- Without this, only the last worker's result would survive (race condition)

### State Transformation (Critical)
When adding a compiled subgraph as a node in the parent graph:
- Parent state → `WorkerInput` (in `dispatch_workers`)
- Subgraph executes with `WorkerState`
- Subgraph returns `WorkerOutput`
- Parent receives `WorkerOutput` as dict, merges into `worker_results` via reducer

---

## Practice Exercise

Use the original notebook (`Orchestrator-Worker with Send API.ipynb`) to:
1. Rebuild this from scratch
2. Try variations:
   - Change number of subtopics (dynamic from LLM)
   - Add a "critic" worker that reviews other workers' output
   - Stream results as they arrive (use `.stream()` instead of `.invoke()`)
   - Add persistence with `MemorySaver` checkpointing

---

## Quick Test (Uncomment to Run)

```python
# report = run_research("AI Engineering")
# print(report)
```